# Skill-Based Gig Matching for GigLink Using TF-IDF, Latent Semantic Analysis, and Optimal Assignment

MCS 306 – Introduction to Artificial Intelligence · TWM 301 – Technical Writing in Mathematics
Bulacan State University · BSM CS 3B – G2

**Authors:** Kristiana T. Evangelista, Thealyne G. Jacinto, Jammele E. Perez, Tifany B. Santos, Justine S. Segubiense

## Data

| File | Source |
|---|---|
| `upwork-jobs.csv` | [Upwork Job Postings Dataset 2024 (50K Records)](https://www.kaggle.com/datasets/asaniczka/upwork-job-postings-dataset-2024-50k-records), asaniczka, Kaggle |
| `Resume.csv` | [Resume Dataset](https://www.kaggle.com/datasets/snehaanbhawal/resume-dataset), S. Bhawal, Kaggle (inside the zip at `Resume/Resume.csv`) |

Download both files from Kaggle and put them in **My Drive**, then choose **Runtime → Run all**.
If they are not found in Drive, the next cell asks you to upload them.

In [ ]:
import glob, os, shutil

os.makedirs("/content/giglink/giglink_ai", exist_ok=True)
os.makedirs("/content/giglink/data/raw", exist_ok=True)
os.chdir("/content/giglink")

from google.colab import drive, files
drive.mount("/content/drive")
for name in ["upwork-jobs.csv", "Resume.csv"]:
    found = glob.glob(f"/content/drive/MyDrive/**/{name}", recursive=True)
    if found:
        shutil.copy(found[0], f"data/raw/{name}")
    else:
        print(f"Upload {name}")
        for f in files.upload():
            shutil.move(f, f"data/raw/{name}")
print(os.listdir("data/raw"))

## Code

In [ ]:
%%writefile data/category_map.csv
upwork_category,primary_resume_category,related_resume_categories
Full Stack Development,INFORMATION-TECHNOLOGY,ENGINEERING
Front-End Development,INFORMATION-TECHNOLOGY,ENGINEERING;DESIGNER
Back-End Development,INFORMATION-TECHNOLOGY,ENGINEERING
Mobile App Development,INFORMATION-TECHNOLOGY,ENGINEERING
Ecommerce Website Development,INFORMATION-TECHNOLOGY,DIGITAL-MEDIA;DESIGNER
CMS Development,INFORMATION-TECHNOLOGY,DIGITAL-MEDIA
Scripting & Automation,INFORMATION-TECHNOLOGY,ENGINEERING
Desktop Software Development,INFORMATION-TECHNOLOGY,ENGINEERING
Business Applications Development,INFORMATION-TECHNOLOGY,ENGINEERING
Video Game Development,INFORMATION-TECHNOLOGY,ARTS
Mobile Game Development,INFORMATION-TECHNOLOGY,ARTS
Systems Administration,INFORMATION-TECHNOLOGY,ENGINEERING
Network Administration,INFORMATION-TECHNOLOGY,ENGINEERING
DevOps Engineering,INFORMATION-TECHNOLOGY,ENGINEERING
Information Security,INFORMATION-TECHNOLOGY,ENGINEERING
Database Administration,INFORMATION-TECHNOLOGY,ENGINEERING
Database Development,INFORMATION-TECHNOLOGY,ENGINEERING
Data Engineering,INFORMATION-TECHNOLOGY,ENGINEERING
Solution Architecture,INFORMATION-TECHNOLOGY,ENGINEERING
Tech Support,INFORMATION-TECHNOLOGY,BPO
Manual Testing,INFORMATION-TECHNOLOGY,ENGINEERING
Machine Learning,INFORMATION-TECHNOLOGY,ENGINEERING
Deep Learning,INFORMATION-TECHNOLOGY,ENGINEERING
Data Analytics,INFORMATION-TECHNOLOGY,FINANCE;CONSULTANT
Data Visualization,INFORMATION-TECHNOLOGY,FINANCE;CONSULTANT
Technical Writing,INFORMATION-TECHNOLOGY,ENGINEERING
Graphic Design,DESIGNER,ARTS;DIGITAL-MEDIA
Logo Design,DESIGNER,ARTS;DIGITAL-MEDIA
Brand Identity Design,DESIGNER,ARTS;DIGITAL-MEDIA
Presentation Design,DESIGNER,DIGITAL-MEDIA
Packaging Design,DESIGNER,ARTS
Editorial Design,DESIGNER,ARTS;DIGITAL-MEDIA
Illustration,DESIGNER,ARTS
Image Editing,DESIGNER,DIGITAL-MEDIA;ARTS
UX/UI Design,DESIGNER,INFORMATION-TECHNOLOGY;DIGITAL-MEDIA
Web Design,DESIGNER,INFORMATION-TECHNOLOGY;DIGITAL-MEDIA
Product & Industrial Design,DESIGNER,ENGINEERING
3D Modeling & Rendering,DESIGNER,ARTS;ENGINEERING
Interior Design,DESIGNER,CONSTRUCTION;ARTS
Fashion Design,APPAREL,DESIGNER;ARTS
Video Editing,DIGITAL-MEDIA,ARTS;DESIGNER
Video Production,DIGITAL-MEDIA,ARTS
Videography,DIGITAL-MEDIA,ARTS
Motion Graphics,DIGITAL-MEDIA,DESIGNER;ARTS
2D Animation,DIGITAL-MEDIA,ARTS;DESIGNER
3D Animation,DIGITAL-MEDIA,ARTS;DESIGNER
Social Media Marketing,DIGITAL-MEDIA,PUBLIC-RELATIONS;SALES;BUSINESS-DEVELOPMENT
Social Media Strategy,DIGITAL-MEDIA,PUBLIC-RELATIONS;BUSINESS-DEVELOPMENT
Community Management,DIGITAL-MEDIA,PUBLIC-RELATIONS
Content Strategy,DIGITAL-MEDIA,PUBLIC-RELATIONS
SEO,DIGITAL-MEDIA,INFORMATION-TECHNOLOGY;BUSINESS-DEVELOPMENT
Search Engine Marketing,DIGITAL-MEDIA,BUSINESS-DEVELOPMENT;SALES
Email Marketing,DIGITAL-MEDIA,SALES;BUSINESS-DEVELOPMENT
Marketing Automation,DIGITAL-MEDIA,INFORMATION-TECHNOLOGY;BUSINESS-DEVELOPMENT
Campaign Management,DIGITAL-MEDIA,PUBLIC-RELATIONS;BUSINESS-DEVELOPMENT
Marketing Strategy,BUSINESS-DEVELOPMENT,DIGITAL-MEDIA;SALES;CONSULTANT
Market Research,BUSINESS-DEVELOPMENT,CONSULTANT;SALES
Public Relations,PUBLIC-RELATIONS,DIGITAL-MEDIA;BUSINESS-DEVELOPMENT
Marketing Copywriting,PUBLIC-RELATIONS,DIGITAL-MEDIA
Lead Generation,SALES,BUSINESS-DEVELOPMENT;BPO
Sales & Business Development,SALES,BUSINESS-DEVELOPMENT
Telemarketing,SALES,BPO
"Email, Phone & Chat Support",BPO,SALES
Customer Success,BPO,SALES
Accounting,ACCOUNTANT,FINANCE;BANKING
Bookkeeping,ACCOUNTANT,FINANCE;BANKING
Tax Preparation,ACCOUNTANT,FINANCE
Financial Analysis & Modeling,FINANCE,ACCOUNTANT;BANKING;CONSULTANT
Business & Corporate Law,ADVOCATE,CONSULTANT
Intellectual Property Law,ADVOCATE,
Paralegal Services,ADVOCATE,
Recruiting & Talent Sourcing,HR,BUSINESS-DEVELOPMENT
HR Administration,HR,
Training & Development,HR,TEACHER
Management Consulting,CONSULTANT,BUSINESS-DEVELOPMENT;FINANCE
Business Analysis & Strategy,CONSULTANT,BUSINESS-DEVELOPMENT;INFORMATION-TECHNOLOGY
Digital Project Management,CONSULTANT,INFORMATION-TECHNOLOGY;BUSINESS-DEVELOPMENT
Product Management,CONSULTANT,INFORMATION-TECHNOLOGY;BUSINESS-DEVELOPMENT
Mechanical Engineering,ENGINEERING,AUTOMOBILE;CONSTRUCTION
Electrical Engineering,ENGINEERING,CONSTRUCTION
Electronic Engineering,ENGINEERING,INFORMATION-TECHNOLOGY
Firmware Development,ENGINEERING,INFORMATION-TECHNOLOGY
CAD,ENGINEERING,CONSTRUCTION;DESIGNER
Structural Engineering,CONSTRUCTION,ENGINEERING
Civil Engineering,CONSTRUCTION,ENGINEERING
Architectural Design,CONSTRUCTION,DESIGNER;ENGINEERING
Cartoons & Comics,ARTS,DESIGNER;DIGITAL-MEDIA
Local Photography,ARTS,DIGITAL-MEDIA
Acting,ARTS,DIGITAL-MEDIA
Voice Talent,ARTS,DIGITAL-MEDIA
Music Production,ARTS,DIGITAL-MEDIA
Audio Editing,ARTS,DIGITAL-MEDIA
Language Tutoring,TEACHER,
Mathematics,TEACHER,

In [ ]:
%%writefile giglink_ai/__init__.py
"""GigLink AI: skill-based gig matching experiments (MCS 306 / TWM 301)."""

In [ ]:
%%writefile giglink_ai/real_data.py
"""Real-data loader: Upwork job posts (gigs) and LiveCareer resumes (seekers).

Two public Kaggle datasets are used (download links in README.md):

* ``data/raw/upwork-jobs.csv``  real Upwork job posts collected from the
  Upwork RSS feed (Dec 2023 - Feb 2024). The client's Upwork category and the
  Upwork skill tags are embedded in the ``description`` field and parsed here.
* ``data/raw/Resume.csv``  2,484 real resumes scraped from LiveCareer, each
  filed under one of 24 job categories (e.g. DESIGNER, ACCOUNTANT).

Ground truth
------------
There is no public record of who was actually hired for an Upwork post. We
therefore judge relevance through a category compatibility table written by
the authors, ``data/category_map.csv``: for every Upwork category it lists the
resume category that does that work (``primary``) and the categories that do
closely related work (``related``). Graded relevance is

    rel(g, s) = 2  if category(s) is the primary category for gig g,
                1  if category(s) is a related category,
                0  otherwise.

The table only uses the two category labels, which the models never see, so
the models are graded against an external judgement and not their own inputs.
"""

from __future__ import annotations

import re
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path(__file__).resolve().parents[1]
RAW = ROOT / "data" / "raw"
GIGS_CSV = RAW / "upwork-jobs.csv"
RESUMES_CSV = RAW / "Resume.csv"
CATEGORY_MAP = ROOT / "data" / "category_map.csv"

_PAY_MARKERS = r"(?:Hourly Range:|Budget:|Posted On:)"
_FIELD_END = r"(?:Skills:|Location Requirement:|Country:|click to apply)"


# ---------------------------------------------------------------------------
# Parsing
# ---------------------------------------------------------------------------
def _job_text(desc: str) -> str:
    return re.split(_PAY_MARKERS, desc, maxsplit=1)[0].replace("\xa0", " ").strip()


def _category(desc: str) -> str:
    m = re.search(r"Category:\s*(.*?)" + _FIELD_END, desc)
    return m.group(1).strip() if m else ""


def _skills(desc: str) -> str:
    m = re.search(r"Skills:(.*?)" + _FIELD_END, desc, re.S)
    if not m:
        return ""
    tags = [t.strip().lower() for t in m.group(1).split(",")]
    return ";".join(dict.fromkeys(t for t in tags if t))


def load_gigs(path: Path = GIGS_CSV) -> pd.DataFrame:
    """Parse the raw Upwork file into one row per job post."""
    d = pd.read_csv(path)
    gigs = pd.DataFrame({
        "gig_id": d["link"].str.extract(r"(?:~|%7E)(\w+)")[0],
        "title": d["title"].str.strip(),
        "category": d["description"].map(_category),
        "required_skills": d["description"].map(_skills),
        "is_hourly": d["is_hourly"].astype(str).eq("True"),
        "hourly_low": d["hourly_low"],
        "hourly_high": d["hourly_high"],
        "budget": d["budget"],
        "country": d["country"].fillna(""),
        "body": d["description"].map(_job_text),
    })
    gigs["description"] = gigs["title"] + ". " + gigs["body"]
    return gigs.drop_duplicates("gig_id").reset_index(drop=True)


def load_resumes(path: Path = RESUMES_CSV) -> pd.DataFrame:
    d = pd.read_csv(path, usecols=["ID", "Resume_str", "Category"])
    bio = d["Resume_str"].fillna("").str.replace(r"\s+", " ", regex=True).str.strip()
    out = pd.DataFrame({"seeker_id": d["ID"].astype(str), "category": d["Category"], "bio": bio})
    return out[out.bio.str.len() >= 200].reset_index(drop=True)  # drops a handful of near-empty files


def load_category_map(path: Path = CATEGORY_MAP) -> dict[str, tuple[str, set[str]]]:
    m = pd.read_csv(path).fillna("")
    return {r.upwork_category: (r.primary_resume_category, {c for c in r.related_resume_categories.split(";") if c})
            for r in m.itertuples()}


# ---------------------------------------------------------------------------
# Seeker skill tags: Upwork skill phrases that occur in the resume text
# ---------------------------------------------------------------------------
def skill_vocabulary(gigs: pd.DataFrame, min_gigs: int = 5) -> set[str]:
    """Upwork skill tags (at most four words) used by at least ``min_gigs`` job posts."""
    counts = gigs.required_skills.str.split(";").explode().value_counts()
    return {t for t, c in counts.items() if c >= min_gigs and len(t) >= 2 and len(t.split()) <= 4}


def _ngrams(text: str, n_max: int) -> set[str]:
    toks = re.findall(r"[a-z0-9+#./&-]+", text.lower())
    toks = [t.strip(".-/") or t for t in toks]
    grams = set()
    for n in range(1, n_max + 1):
        grams.update(" ".join(toks[i : i + n]) for i in range(len(toks) - n + 1))
    return grams


def extract_tags(texts: pd.Series, vocab: set[str]) -> pd.Series:
    """Tag each resume with every vocabulary phrase it contains (exact n-gram match)."""
    n_max = max(len(v.split()) for v in vocab)
    return texts.map(lambda t: ";".join(sorted(_ngrams(t, n_max) & vocab)))


# ---------------------------------------------------------------------------
# Dataset assembly
# ---------------------------------------------------------------------------
@dataclass
class RealConfig:
    n_gigs: int = 400          # gigs sampled per seed (half validation, half test)
    n_seekers: int | None = None  # None = every resume
    min_tags: int = 1          # gigs need at least this many Upwork skill tags
    seed: int = 0


_CACHE: dict = {}


def _base():
    """Parse the raw files once per process (about 10 s)."""
    if "base" not in _CACHE:
        cmap = load_category_map()
        gigs = load_gigs()
        vocab = skill_vocabulary(gigs)
        gigs = gigs[gigs.category.isin(cmap)].reset_index(drop=True)
        seekers = load_resumes()
        seekers["declared_skills"] = extract_tags(seekers.bio, vocab)
        _CACHE["base"] = (gigs, seekers, cmap, vocab)
    return _CACHE["base"]


def relevance_matrix(gigs: pd.DataFrame, seekers: pd.DataFrame, cmap) -> np.ndarray:
    rel = np.zeros((len(gigs), len(seekers)), dtype=int)
    scat = seekers.category.to_numpy()
    for i, cat in enumerate(gigs.category):
        primary, related = cmap[cat]
        rel[i, np.isin(scat, list(related))] = 1
        rel[i, scat == primary] = 2
    return rel


def sample(cfg: RealConfig):
    """Draw one random evaluation set: (seekers, gigs, relevance)."""
    gigs, seekers, cmap, _ = _base()
    rng = np.random.default_rng(cfg.seed)
    pool = gigs[gigs.required_skills.str.count(";") + 1 >= cfg.min_tags]
    g = pool.iloc[rng.choice(len(pool), size=cfg.n_gigs, replace=False)].reset_index(drop=True)
    s = seekers
    if cfg.n_seekers is not None:
        s = seekers.iloc[np.sort(rng.choice(len(seekers), size=cfg.n_seekers, replace=False))].reset_index(drop=True)
    return s, g, relevance_matrix(g, s, cmap)


def data_available() -> bool:
    return GIGS_CSV.exists() and RESUMES_CSV.exists()

In [ ]:
%%writefile giglink_ai/models.py
"""Scoring models that rank seekers for each gig.

Every model returns a score matrix ``S`` of shape (n_gigs, n_seekers) where a
larger ``S[g, s]`` means seeker ``s`` is a better match for gig ``g``.

Models
------
random_scores    : baseline, uniform random scores.
keyword_scores   : count of shared (non-stopword) words, no weighting.
jaccard_scores   : Jaccard similarity of gig tags vs declared seeker tags.
tfidf_scores     : cosine similarity of TF-IDF vectors (vector space model).
lsa_scores       : cosine similarity after truncated SVD (latent semantics).
hybrid_scores    : weighted combination of text similarity and Jaccard.
"""

from __future__ import annotations

import numpy as np
import pandas as pd
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.preprocessing import normalize



def split_set(s) -> set[str]:
    """Parse a ';'-separated tag string into a set (empty/NaN -> empty set)."""
    if not isinstance(s, str) or not s:
        return set()
    return {t.strip() for t in s.split(";") if t.strip()}


def random_scores(n_gigs: int, n_seekers: int, seed: int = 0) -> np.ndarray:
    return np.random.default_rng(seed).random((n_gigs, n_seekers))


def keyword_scores(gig_text: pd.Series, seeker_text: pd.Series) -> np.ndarray:
    """Number of distinct words shared by the gig and the seeker bio."""
    vec = CountVectorizer(stop_words="english", binary=True)
    vec.fit(pd.concat([gig_text, seeker_text]))
    G = vec.transform(gig_text)
    S = vec.transform(seeker_text)
    return (G @ S.T).toarray().astype(float)


def jaccard_scores(gig_tags: pd.Series, seeker_tags: pd.Series) -> np.ndarray:
    """J(A, B) = |A ∩ B| / |A ∪ B| between required and declared skill tags.

    Vectorised with binary tag matrices A (gigs x skills) and B (seekers x
    skills): |A ∩ B| = A B^T and |A ∪ B| = |A| + |B| - |A ∩ B|.
    """
    vocab = sorted(set().union(*map(split_set, gig_tags), *map(split_set, seeker_tags)))
    col = {t: i for i, t in enumerate(vocab)}

    def binary(tags: pd.Series) -> np.ndarray:
        M = np.zeros((len(tags), len(vocab)))
        for r, x in enumerate(tags):
            for t in split_set(x):
                M[r, col[t]] = 1.0
        return M

    A, B = binary(gig_tags), binary(seeker_tags)
    inter = A @ B.T
    union = A.sum(1)[:, None] + B.sum(1)[None, :] - inter
    return np.divide(inter, union, out=np.zeros_like(inter), where=union > 0)


def fit_tfidf(gig_text: pd.Series, seeker_text: pd.Series):
    """Fit TF-IDF on the joint corpus and return (vectorizer, G, S).

    Weights follow w_{t,d} = (1 + log tf_{t,d}) * (1 + log((1+N)/(1+df_t)))
    and each document vector is L2-normalised, so a dot product is a cosine.
    """
    vec = TfidfVectorizer(stop_words="english", sublinear_tf=True, norm="l2")
    vec.fit(pd.concat([gig_text, seeker_text]))
    return vec, vec.transform(gig_text), vec.transform(seeker_text)


def tfidf_scores(gig_text: pd.Series, seeker_text: pd.Series) -> np.ndarray:
    _, G, S = fit_tfidf(gig_text, seeker_text)
    return (G @ S.T).toarray()


def lsa_scores(
    gig_text: pd.Series,
    seeker_text: pd.Series,
    k: int = 40,
    n_iter: int = 7,
    seed: int = 0,
) -> np.ndarray:
    """Latent Semantic Analysis similarity.

    The TF-IDF matrix X (documents x terms) is approximated by a rank-k
    truncated SVD X ≈ U_k Σ_k V_k^T. Documents are projected to the k-dim
    latent space (rows of U_k Σ_k), re-normalised, and compared by cosine.
    ``n_iter`` is the number of power iterations of the randomized SVD.
    """
    _, G, S = fit_tfidf(gig_text, seeker_text)
    from scipy.sparse import vstack

    X = vstack([G, S])
    k = min(k, X.shape[1] - 1)
    svd = TruncatedSVD(n_components=k, n_iter=n_iter, random_state=seed)
    Z = normalize(svd.fit_transform(X))
    Zg, Zs = Z[: G.shape[0]], Z[G.shape[0] :]
    return Zg @ Zs.T


def rating_feature(ratings: pd.Series, prior: float = 3.5) -> np.ndarray:
    """Map ratings in [1,5] to [0,1]; cold-start seekers get a neutral prior."""
    r = ratings.fillna(prior).to_numpy(dtype=float)
    return (r - 1.0) / 4.0


def hybrid_scores(
    text_sim: np.ndarray,
    tag_sim: np.ndarray,
    rating01: np.ndarray,
    w_text: float = 0.6,
    w_rating: float = 0.05,
) -> np.ndarray:
    """Hybrid score H = w_text*text + (1-w_text)*tag + w_rating*rating.

    The text and tag terms form a convex combination; the rating term is a
    small additive bonus so that reputation breaks ties between equally
    skilled seekers without dominating skill fit.
    """
    return w_text * text_sim + (1.0 - w_text) * tag_sim + w_rating * rating01[None, :]


def apply_feasibility(scores: np.ndarray, feasible: np.ndarray) -> np.ndarray:
    """Push infeasible pairs to the bottom of every ranking (hard constraint)."""
    out = scores.astype(float).copy()
    out[~feasible] = -np.inf
    return out

In [ ]:
%%writefile giglink_ai/metrics.py
"""Ranking metrics: Precision@k, Recall@k, nDCG@k and MRR.

All functions take a score matrix ``scores`` (n_gigs, n_seekers) and a graded
relevance matrix ``rel`` of the same shape, and return the mean over gigs.
A seeker counts as *relevant* when rel >= ``threshold`` (default 2, meaning
the resume belongs to the primary matching category for the gig).
"""

from __future__ import annotations

import numpy as np


def _topk(scores: np.ndarray, k: int) -> np.ndarray:
    """Indices of the k highest scores per row, best first (ties: lower index)."""
    order = np.argsort(-scores, axis=1, kind="stable")
    return order[:, :k]


def precision_at_k(scores, rel, k=5, threshold=2):
    top = _topk(scores, k)
    hits = np.take_along_axis(rel, top, axis=1) >= threshold
    return float(hits.mean())


def recall_at_k(scores, rel, k=10, threshold=2):
    top = _topk(scores, k)
    hits = (np.take_along_axis(rel, top, axis=1) >= threshold).sum(axis=1)
    n_rel = (rel >= threshold).sum(axis=1)
    mask = n_rel > 0
    return float((hits[mask] / n_rel[mask]).mean())


def ndcg_at_k(scores, rel, k=10):
    """nDCG@k with gain 2^rel - 1 and discount 1/log2(rank + 1)."""
    discounts = 1.0 / np.log2(np.arange(2, k + 2))
    top = _topk(scores, k)
    gains = 2.0 ** np.take_along_axis(rel, top, axis=1) - 1
    dcg = (gains * discounts).sum(axis=1)
    ideal = 2.0 ** (-np.sort(-rel, axis=1)[:, :k]) - 1
    idcg = (ideal * discounts).sum(axis=1)
    mask = idcg > 0
    return float((dcg[mask] / idcg[mask]).mean())


def mrr(scores, rel, threshold=2):
    """Mean reciprocal rank of the first relevant seeker."""
    order = np.argsort(-scores, axis=1, kind="stable")
    is_rel = np.take_along_axis(rel, order, axis=1) >= threshold
    has = is_rel.any(axis=1)
    first = is_rel.argmax(axis=1)
    return float((1.0 / (first[has] + 1)).mean())


def all_metrics(scores, rel) -> dict[str, float]:
    return {
        "P@5": precision_at_k(scores, rel, 5),
        "R@100": recall_at_k(scores, rel, 100),
        "nDCG@10": ndcg_at_k(scores, rel, 10),
        "MRR": mrr(scores, rel),
    }

In [ ]:
%%writefile giglink_ai/assignment.py
"""One-to-one assignment of seekers to gigs.

Ranking answers "who is good for this gig?". When many gigs are posted at
once, the platform should also avoid recommending the same top seeker to
every client. We model this as the linear assignment problem

    maximise   sum_{g,s} H[g,s] * x[g,s]
    subject to sum_s x[g,s] <= 1   (each gig gets at most one seeker)
               sum_g x[g,s] <= 1   (each seeker takes at most one gig)
               x[g,s] = 0 if (g,s) is infeasible,   x in {0,1}

and solve it exactly with SciPy's ``linear_sum_assignment`` (a shortest
augmenting path variant of the Hungarian / Jonker-Volgenant method). A greedy
heuristic and an "independent top-1" policy are used for comparison.
"""

from __future__ import annotations

import numpy as np
from scipy.optimize import linear_sum_assignment

# Score given to forbidden pairs. Far below any real score (which lie in
# roughly [0, 1.1]) so the optimiser never picks them when a feasible choice
# exists; any such pair that is still chosen is dropped afterwards.
_FORBIDDEN = -1e6


def optimal_assignment(scores: np.ndarray, feasible: np.ndarray) -> list[tuple[int, int]]:
    H = np.where(feasible, scores, _FORBIDDEN)
    rows, cols = linear_sum_assignment(H, maximize=True)
    return [(g, s) for g, s in zip(rows, cols) if feasible[g, s]]


def greedy_assignment(scores: np.ndarray, feasible: np.ndarray) -> list[tuple[int, int]]:
    """Repeatedly take the highest-scoring remaining feasible pair."""
    H = np.where(feasible, scores, -np.inf)
    order = np.argsort(-H, axis=None, kind="stable")
    used_g, used_s, pairs = set(), set(), []
    n_s = H.shape[1]
    for flat in order:
        g, s = divmod(int(flat), n_s)
        if not np.isfinite(H[g, s]):
            break
        if g in used_g or s in used_s:
            continue
        pairs.append((g, s))
        used_g.add(g)
        used_s.add(s)
    return pairs


def independent_top1(scores: np.ndarray, feasible: np.ndarray) -> list[tuple[int, int]]:
    """Every gig takes its own best seeker; seekers may be double-booked."""
    H = np.where(feasible, scores, -np.inf)
    best = H.argmax(axis=1)
    return [(g, int(s)) for g, s in enumerate(best) if np.isfinite(H[g, s])]


def evaluate_assignment(pairs, scores, rel) -> dict[str, float]:
    """Summarise an assignment using predicted scores and TRUE relevance."""
    n_gigs = rel.shape[0]
    seekers = [s for _, s in pairs]
    double_booked = len(seekers) - len(set(seekers))
    return {
        "predicted_utility": float(sum(scores[g, s] for g, s in pairs)),
        "true_relevance_sum": float(sum(rel[g, s] for g, s in pairs)),
        "good_match_rate": float(sum(rel[g, s] >= 2 for g, s in pairs) / n_gigs),
        "gigs_filled": len(pairs),
        "double_booked": double_booked,
    }

In [ ]:
%%writefile giglink_ai/plotting.py
"""Shared matplotlib style so every figure in the paper looks consistent."""

from __future__ import annotations

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt  # noqa: E402

# Fixed categorical order (colour-blind validated). Each method keeps the same
# colour in every figure, so readers can track it across the paper.
PALETTE = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]
METHOD_COLORS = {
    "Random": "#8a8984",
    "Keyword": PALETTE[3],
    "Jaccard (tags)": PALETTE[1],
    "TF-IDF": PALETTE[2],
    "LSA": PALETTE[6],
    "Hybrid": PALETTE[0],
}
MARKERS = {"Random": "x", "Keyword": "s", "Jaccard (tags)": "^", "TF-IDF": "o", "LSA": "D", "Hybrid": "*"}


def setup() -> None:
    plt.rcParams.update(
        {
            "figure.dpi": 150,
            "savefig.dpi": 300,
            "savefig.bbox": "tight",
            "font.size": 9,
            "axes.titlesize": 10,
            "axes.labelsize": 9,
            "axes.spines.top": False,
            "axes.spines.right": False,
            "axes.edgecolor": "#52514e",
            "axes.grid": True,
            "axes.axisbelow": True,
            "grid.color": "#e4e3df",
            "grid.linewidth": 0.6,
            "lines.linewidth": 2.0,
            "lines.markersize": 5,
            "legend.frameon": False,
            "legend.fontsize": 8,
        }
    )

In [ ]:
%%writefile giglink_ai/make_tables.py
"""Turn the CSV files in results/ into LaTeX tables in paper/tables/.

Generating the tables from the result files (instead of typing numbers by
hand) keeps the paper and the experiments consistent. Run after
``python -m giglink_ai.experiments``::

    python -m giglink_ai.make_tables
"""

from __future__ import annotations

import json
from pathlib import Path

import pandas as pd

ROOT = Path(__file__).resolve().parents[1]
RES, OUT = ROOT / "results", ROOT / "paper" / "tables"
METRICS = ["P@5", "R@100", "nDCG@10", "MRR"]


def _fmt(mu, ci, bold=False):
    s = f"{mu:.3f} $\\pm$ {ci:.3f}"
    return f"\\textbf{{{mu:.3f}}} $\\pm$ {ci:.3f}" if bold else s


def _metric_table(df, key, order, caption, label):
    df = df.set_index(key).loc[order]
    best = {m: df[f"{m}_mean"].max() for m in METRICS}
    lines = [
        "\\begin{table}[htbp]", "\\centering", "\\small", f"\\caption{{{caption}}}", f"\\label{{{label}}}",
        "\\begin{tabular}{l" + "c" * len(METRICS) + "}", "\\toprule",
        "Method & " + " & ".join(m.replace("@", "@") for m in METRICS) + " \\\\", "\\midrule",
    ]
    for name, row in df.iterrows():
        name = name.replace("w_text", "$\\alpha$")
        cells = [_fmt(row[f"{m}_mean"], row[f"{m}_ci95"], abs(row[f"{m}_mean"] - best[m]) < 1e-12) for m in METRICS]
        lines.append(f"{name} & " + " & ".join(cells) + " \\\\")
    lines += ["\\bottomrule", "\\end{tabular}", "\\end{table}"]
    return "\n".join(lines) + "\n"


def main():
    OUT.mkdir(parents=True, exist_ok=True)
    main_sum = pd.read_csv(RES / "main_metrics_summary.csv")
    meta = json.loads((RES / "run_metadata.json").read_text())
    order = ["Random", "Keyword", "Jaccard (tags)", "TF-IDF", "LSA", "Hybrid"]
    (OUT / "main_metrics.tex").write_text(_metric_table(
        main_sum, "method", order,
        "Test-set ranking quality on real Upwork job posts and real resumes (mean $\\pm$ half-width of the 95\\% confidence interval over "
        f"{int(main_sum.n_seeds.iloc[0])} random samples; 200 test gigs $\\times$ {meta['data']['seekers']:,} resumes per sample). "
        "Best value per column in bold.", "tab:main"))

    ab = pd.read_csv(RES / "ablation_summary.csv")
    (OUT / "ablation.tex").write_text(_metric_table(
        ab, "variant", list(ab.variant),
        f"Ablation of the hybrid model (test gigs, {int(ab.n_seeds.iloc[0])} samples, mean $\\pm$ 95\\% CI). "
        "$\\alpha$ is the weight on text similarity; $1-\\alpha$ goes to tag overlap.", "tab:ablation"))

    asg = pd.read_csv(RES / "assignment_summary.csv")
    lines = ["\\begin{table}[htbp]", "\\centering", "\\small",
             f"\\caption{{Simultaneous assignment of all {meta['config']['n_gigs']} sampled gigs (mean over {int(main_sum.n_seeds.iloc[0])} samples). "
             "\\emph{True relevance} sums the category-based relevance of the chosen pairs; \\emph{good match rate} is the share of the gigs "
             "that receive a resume from the primary matching category ($\\mathrm{rel}=2$).}",
             "\\label{tab:assign}", "\\resizebox{\\linewidth}{!}{%", "\\begin{tabular}{llrrrrr}", "\\toprule",
             "Market & Policy & Pred.\\ utility & True relevance & Good match rate & Gigs filled & Double-booked \\\\", "\\midrule"]
    for i, r in asg.iterrows():
        market = r.market if i % 3 == 0 else ""
        lines.append(f"{market} & {r.policy} & {r.predicted_utility:.2f} & {r.true_relevance_sum:.2f} & "
                     f"{r.good_match_rate:.3f} & {r.gigs_filled:.1f} & {r.double_booked:.1f} \\\\")
        if i == 2:
            lines.append("\\midrule")
    lines += ["\\bottomrule", "\\end{tabular}}", "\\end{table}"]
    (OUT / "assignment.tex").write_text("\n".join(lines) + "\n")

    sig = pd.read_csv(RES / "significance_tests.csv")
    lines = ["\\begin{table}[htbp]", "\\centering", "\\small",
             f"\\caption{{Paired comparisons over the same {int(main_sum.n_seeds.iloc[0])} random samples (Hybrid minus baseline).}}", "\\label{tab:sig}",
             "\\resizebox{\\linewidth}{!}{%", "\\begin{tabular}{llrrrr}", "\\toprule",
             "Baseline & Metric & Mean diff. & Paired $t$ ($p$) & Wilcoxon $p$ & Hybrid wins \\\\", "\\midrule"]
    for _, r in sig.iterrows():
        lines.append(f"{r.b} & {r.metric} & {r.mean_diff:+.3f} & {r.t_stat:.1f} ({r.t_p:.1e}) & {r.wilcoxon_p:.1e} & {r.a_wins}/{r.n} \\\\")
    lines += ["\\bottomrule", "\\end{tabular}}", "\\end{table}"]
    (OUT / "significance.tex").write_text("\n".join(lines) + "\n")

    sc = pd.read_csv(RES / "runtime_scaling.csv").groupby("n").median(numeric_only=True)
    slopes = meta["runtime_loglog_slopes_n>=400"]
    cols = ["TF-IDF scoring", "Jaccard scoring", "Hungarian assignment", "Greedy assignment"]
    lines = ["\\begin{table}[htbp]", "\\centering", "\\small",
             "\\caption{Wall-clock time in seconds (median of 3 runs) for $n$ gigs and $n$ seekers. The last row is the slope of a "
             "least-squares line fitted to $\\log t$ versus $\\log n$ for $n \\ge 400$ (empirical complexity exponent).}",
             "\\label{tab:runtime}", "\\begin{tabular}{rrrrr}", "\\toprule",
             "$n$ & TF-IDF scoring & Jaccard scoring & Hungarian & Greedy \\\\", "\\midrule"]
    for n, r in sc.iterrows():
        lines.append(f"{n} & " + " & ".join(f"{r[c]:.4f}" for c in cols) + " \\\\")
    lines += ["\\midrule", "slope & " + " & ".join(f"{slopes[c]:.2f}" for c in cols) + " \\\\",
              "\\bottomrule", "\\end{tabular}", "\\end{table}"]
    (OUT / "runtime.tex").write_text("\n".join(lines) + "\n")

    svd = pd.read_csv(RES / "svd_convergence.csv").groupby("n_iter").mean(numeric_only=True)
    lines = ["\\begin{table}[htbp]", "\\centering", "\\small",
             f"\\caption{{Randomized SVD versus exact ARPACK singular values ($k={meta['svd']['k']}$, "
             f"matrix {meta['svd']['matrix_shape'][0]}$\\times${meta['svd']['matrix_shape'][1]}, mean of 5 random starts).}}",
             "\\label{tab:svd}", "\\begin{tabular}{rccc}", "\\toprule",
             "$q$ & max rel.\\ error & mean rel.\\ error & explained variance \\\\", "\\midrule"]
    for q, r in svd.iterrows():
        if q in (0, 1, 2, 4, 6, 7, 8, 10):
            lines.append(f"{q} & {r.max_rel_err:.2e} & {r.mean_rel_err:.2e} & {r.explained_var:.4f} \\\\")
    lines += ["\\bottomrule", "\\end{tabular}", "\\end{table}"]
    (OUT / "svd.tex").write_text("\n".join(lines) + "\n")
    d = meta["data"]
    macros = {
        "DataGigsRaw": f"{d['gigs_raw']:,}", "DataGigsMapped": f"{d['gigs_after_mapping']:,}",
        "DataMappedCats": str(d["upwork_categories_mapped"]), "DataSeekers": f"{d['seekers']:,}",
        "DataVocab": f"{d['skill_vocabulary']:,}", "DataGigTags": f"{d['gig_tags_median']:.0f}",
        "DataSeekerTags": f"{d['seeker_tags_median']:.0f}", "DataResumeWords": f"{d['resume_words_median']:,.0f}",
        "DataRelTwo": f"{d['sample0_mean_rel2_per_gig']:.0f}",
        "DataRelTwoPct": f"{100 * d['sample0_mean_rel2_per_gig'] / d['seekers']:.1f}\\%",
        "DataRuntimeMin": f"{meta['total_runtime_seconds'] / 60:.0f}",
    }
    (OUT / "data_macros.tex").write_text(
        "% Generated by giglink_ai/make_tables.py from results/run_metadata.json\n"
        + "".join(f"\\newcommand{{\\{k}}}{{{v}}}\n" for k, v in macros.items()))
    print(f"wrote tables to {OUT}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile giglink_ai/experiments.py
"""All experiments reported in the paper, on real Upwork job posts and real resumes.

Put the two Kaggle files in data/raw/ first (see README.md), then run from the
repository root::

    python -m giglink_ai.experiments            # full run (a few minutes)
    python -m giglink_ai.experiments --quick    # fewer seeds, for a smoke test

Outputs
-------
results/*.csv|json   every number quoted in the paper
figures/*.pdf|png    every figure in the paper
paper/tables/*.tex   LaTeX tables (via make_tables)
"""

from __future__ import annotations

import argparse
import json
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats
from scipy.sparse import vstack
from scipy.sparse.linalg import svds
from sklearn.decomposition import TruncatedSVD

from . import plotting
from .assignment import evaluate_assignment, greedy_assignment, independent_top1, optimal_assignment
from .metrics import all_metrics, ndcg_at_k
from .models import (
    fit_tfidf,
    hybrid_scores,
    jaccard_scores,
    keyword_scores,
    lsa_scores,
    random_scores,
    split_set,
    tfidf_scores,
)
from .real_data import RealConfig, _base, data_available, load_gigs, sample

ROOT = Path(__file__).resolve().parents[1]
RESULTS, FIGS = ROOT / "results", ROOT / "figures"

# Hyper-parameter grid searched on the VALIDATION gigs only.
W_TEXT_GRID = np.round(np.arange(0.0, 1.01, 0.1), 2)
LSA_K = 40
SVD_ITERS = 7
N_SCARCE = 300  # seekers in the "scarce" assignment market


# ---------------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------------
def base_scores(seekers, gigs, seed, k=LSA_K):
    """Every base score matrix (n_gigs x n_seekers) for one sample."""
    return {
        "Random": random_scores(len(gigs), len(seekers), seed=seed + 999),
        "Keyword": keyword_scores(gigs.description, seekers.bio),
        "Jaccard (tags)": jaccard_scores(gigs.required_skills, seekers.declared_skills),
        "TF-IDF": tfidf_scores(gigs.description, seekers.bio),
        "LSA": lsa_scores(gigs.description, seekers.bio, k=k, n_iter=SVD_ITERS, seed=seed),
    }


def tune_hybrid(base, rel, idx):
    """Grid search (text source, w_text) maximising validation nDCG@10."""
    zero = np.zeros(base["TF-IDF"].shape[1])
    best, table = None, []
    for text in ("TF-IDF", "LSA"):
        for wt in W_TEXT_GRID:
            H = hybrid_scores(base[text], base["Jaccard (tags)"], zero, wt, 0.0)
            score = ndcg_at_k(H[idx], rel[idx], 10)
            table.append({"text": text, "w_text": wt, "val_nDCG@10": score})
            if best is None or score > best[0] + 1e-12:
                best = (score, text, float(wt))
    return best, pd.DataFrame(table)


def hybrid(base, text, wt):
    return hybrid_scores(base[text], base["Jaccard (tags)"], np.zeros(base[text].shape[1]), wt, 0.0)


def mean_ci(x):
    x = np.asarray(x, dtype=float)
    half = stats.t.ppf(0.975, len(x) - 1) * x.std(ddof=1) / np.sqrt(len(x)) if len(x) > 1 else 0.0
    return x.mean(), x.std(ddof=1) if len(x) > 1 else 0.0, half


# ---------------------------------------------------------------------------
# E1  Main comparison + E5 weight grid + E6 assignment, over many random samples
# ---------------------------------------------------------------------------
def run_main(seeds):
    rows, chosen, assign_rows, ablation_rows, per_cat = [], [], [], [], []
    grid_seed0 = None
    for seed in seeds:
        seekers, gigs, rel = sample(RealConfig(seed=seed))
        base = base_scores(seekers, gigs, seed)
        perm = np.random.default_rng(seed).permutation(len(gigs))
        val, test = perm[: len(gigs) // 2], perm[len(gigs) // 2 :]

        (val_score, text, wt), grid = tune_hybrid(base, rel, val)
        if seed == seeds[0]:
            grid_seed0 = grid
        chosen.append({"seed": seed, "text": text, "w_text": wt, "val_nDCG@10": val_score})
        H = hybrid(base, text, wt)
        methods = dict(base, Hybrid=H)

        for name, sc in methods.items():
            rows.append({"seed": seed, "method": name, **all_metrics(sc[test], rel[test])})

        # nDCG@10 of each test gig, to see which kinds of work are easy or hard.
        from .metrics import ndcg_at_k as _n
        for g in test:
            per_cat.append({"seed": seed, "category": gigs.category[g],
                            "Hybrid": _n(H[[g]], rel[[g]], 10), "TF-IDF": _n(base["TF-IDF"][[g]], rel[[g]], 10),
                            "Jaccard (tags)": _n(base["Jaccard (tags)"][[g]], rel[[g]], 10)})

        # Ablations of the hybrid (test gigs).
        other = "LSA" if text == "TF-IDF" else "TF-IDF"
        ab = {
            f"Hybrid (tuned: {text})": H,
            f"Hybrid with {other} text, re-tuned": hybrid(base, other, float(grid[grid.text == other].sort_values("val_nDCG@10").iloc[-1].w_text)),
            "text only ($\\alpha = 1$)": hybrid(base, text, 1.0),
            "tags only ($\\alpha = 0$)": hybrid(base, text, 0.0),
            "equal weights ($\\alpha = 0.5$)": hybrid(base, text, 0.5),
        }
        for name, sc in ab.items():
            key = "Hybrid (tuned)" if name.startswith("Hybrid (tuned") else ("Hybrid, other text model" if name.startswith("Hybrid with") else name)
            ablation_rows.append({"seed": seed, "variant": key, **all_metrics(sc[test], rel[test])})

        # Assignment of ALL sampled gigs, in two market conditions.
        feas = np.ones_like(H, dtype=bool)
        sub = np.sort(np.random.default_rng(seed + 7).choice(len(seekers), N_SCARCE, replace=False))
        for market, cols in ((f"surplus ({len(seekers)} seekers)", np.arange(len(seekers))), (f"scarce ({N_SCARCE} seekers)", sub)):
            Hm, Fm, Rm = H[:, cols], feas[:, cols], rel[:, cols]
            for pol, fn in (("Optimal (Hungarian)", optimal_assignment), ("Greedy", greedy_assignment), ("Independent top-1", independent_top1)):
                t0 = time.perf_counter()
                pairs = fn(Hm, Fm)
                dt = time.perf_counter() - t0
                assign_rows.append({"seed": seed, "market": market, "policy": pol, "seconds": dt, **evaluate_assignment(pairs, Hm, Rm)})
        print(f"  seed {seed}: hybrid text={text} w_text={wt}", flush=True)

    return (pd.DataFrame(rows), pd.DataFrame(chosen), pd.DataFrame(assign_rows),
            pd.DataFrame(ablation_rows), grid_seed0, pd.DataFrame(per_cat))


def summarise(df, by, metrics):
    out = []
    for key, g in df.groupby(by, sort=False):
        row = {by: key, "n_seeds": len(g)}
        for m in metrics:
            mu, sd, half = mean_ci(g[m])
            row[f"{m}_mean"], row[f"{m}_std"], row[f"{m}_ci95"] = mu, sd, half
        out.append(row)
    return pd.DataFrame(out)


def paired_tests(df, a="Hybrid", b="Jaccard (tags)", metric="nDCG@10"):
    x = df[df.method == a].sort_values("seed")[metric].to_numpy()
    y = df[df.method == b].sort_values("seed")[metric].to_numpy()
    t = stats.ttest_rel(x, y)
    w = stats.wilcoxon(x, y)
    return {"a": a, "b": b, "metric": metric, "mean_diff": float((x - y).mean()),
            "t_stat": float(t.statistic), "t_p": float(t.pvalue), "wilcoxon_stat": float(w.statistic),
            "wilcoxon_p": float(w.pvalue), "a_wins": int((x > y).sum()), "n": int(len(x))}


# ---------------------------------------------------------------------------
# E2  LSA dimension sweep   E3  randomized SVD convergence
# ---------------------------------------------------------------------------
def run_lsa_sweep(seeds, ks=(10, 20, 40, 80, 160, 320)):
    rows = []
    for seed in seeds:
        seekers, gigs, rel = sample(RealConfig(seed=seed))
        tf = all_metrics(tfidf_scores(gigs.description, seekers.bio), rel)["nDCG@10"]
        for k in ks:
            sc = lsa_scores(gigs.description, seekers.bio, k=k, n_iter=SVD_ITERS, seed=seed)
            rows.append({"seed": seed, "k": k, "nDCG@10": all_metrics(sc, rel)["nDCG@10"], "tfidf_nDCG@10": tf})
    return pd.DataFrame(rows)


def run_svd_convergence(seed=0, k=LSA_K, iters=range(0, 11)):
    """Relative error of randomized-SVD singular values vs. an exact solver."""
    seekers, gigs, _ = sample(RealConfig(seed=seed))
    _, G, S = fit_tfidf(gigs.description, seekers.bio)
    X = vstack([G, S]).tocsr().astype(float)
    exact = np.sort(svds(X, k=k, tol=0, random_state=seed)[1])[::-1]  # ARPACK, tol=0 -> machine precision
    fro = np.sqrt(X.multiply(X).sum())
    rows = []
    for n_iter in iters:
        for rep in range(5):
            svd = TruncatedSVD(n_components=k, n_iter=n_iter, random_state=100 + rep).fit(X)
            sv = svd.singular_values_
            rows.append({"n_iter": n_iter, "rep": rep,
                         "max_rel_err": float(np.max(np.abs(sv - exact) / exact)),
                         "mean_rel_err": float(np.mean(np.abs(sv - exact) / exact)),
                         "explained_var": float(svd.explained_variance_ratio_.sum())})
    meta = {"matrix_shape": list(X.shape), "nnz": int(X.nnz), "k": k,
            "exact_low_rank_fro_err": float(np.sqrt(max(fro**2 - (exact**2).sum(), 0)) / fro),
            "sigma_1": float(exact[0]), "sigma_k": float(exact[-1])}
    return pd.DataFrame(rows), meta


# ---------------------------------------------------------------------------
# E4  Robustness: shorter resumes and missing skill tags (perturbing real data)
# ---------------------------------------------------------------------------
def _truncate(bio: pd.Series, n_words: int | None) -> pd.Series:
    return bio if n_words is None else bio.str.split().str[:n_words].str.join(" ")


def _drop_tags(tags: pd.Series, p: float, rng) -> pd.Series:
    return tags.map(lambda x: ";".join(t for t in sorted(split_set(x)) if rng.random() >= p))


def run_robustness(seeds, params):
    rows = []
    levels = {"resume_words": [25, 50, 100, 200, 400, None], "tag_drop_prob": [0.0, 0.2, 0.4, 0.6, 0.8]}
    for field, values in levels.items():
        for v in values:
            for seed in seeds:
                seekers, gigs, rel = sample(RealConfig(seed=seed))
                seekers = seekers.copy()
                if field == "resume_words":
                    seekers["bio"] = _truncate(seekers.bio, v)
                else:
                    seekers["declared_skills"] = _drop_tags(seekers.declared_skills, v, np.random.default_rng(seed + 31))
                base = base_scores(seekers, gigs, seed)
                base["Hybrid"] = hybrid(base, params["text"], params["w_text"])
                for name, sc in base.items():
                    rows.append({"factor": field, "level": "all" if v is None else v, "seed": seed, "method": name,
                                 "nDCG@10": all_metrics(sc, rel)["nDCG@10"]})
    return pd.DataFrame(rows)


# ---------------------------------------------------------------------------
# E7  Runtime scaling
# ---------------------------------------------------------------------------
def run_scaling(sizes=(100, 200, 400, 800, 1600, 2400), reps=3):
    rows = []
    for n in sizes:
        seekers, gigs, _ = sample(RealConfig(n_gigs=n, n_seekers=n, seed=0))
        feas = np.ones((n, n), dtype=bool)
        for rep in range(reps):
            t0 = time.perf_counter(); sc = tfidf_scores(gigs.description, seekers.bio); t1 = time.perf_counter()
            jac = jaccard_scores(gigs.required_skills, seekers.declared_skills); t2 = time.perf_counter()
            H = hybrid_scores(sc, jac, np.zeros(n), 0.5, 0.0)
            t3 = time.perf_counter(); optimal_assignment(H, feas); t4 = time.perf_counter()
            greedy_assignment(H, feas); t5 = time.perf_counter()
            rows.append({"n": n, "rep": rep, "TF-IDF scoring": t1 - t0, "Jaccard scoring": t2 - t1,
                         "Hungarian assignment": t4 - t3, "Greedy assignment": t5 - t4})
    return pd.DataFrame(rows)


# ---------------------------------------------------------------------------
# Figures
# ---------------------------------------------------------------------------
ORDER = ["Random", "Keyword", "Jaccard (tags)", "TF-IDF", "LSA", "Hybrid"]


def fig_main(summary):
    import matplotlib.pyplot as plt
    fig, axes = plt.subplots(1, 4, figsize=(10, 2.8), sharey=True)
    s = summary.set_index("method").loc[ORDER]
    for ax, m in zip(axes, ["P@5", "R@100", "nDCG@10", "MRR"]):
        y = np.arange(len(ORDER))
        ax.barh(y, s[f"{m}_mean"], xerr=s[f"{m}_ci95"], color=[plotting.METHOD_COLORS[k] for k in ORDER],
                height=0.65, error_kw={"elinewidth": 1, "capsize": 2, "ecolor": "#52514e"})
        for yi, v, h in zip(y, s[f"{m}_mean"], s[f"{m}_ci95"]):
            ax.text(v + h + 0.03, yi, f"{v:.3f}", va="center", fontsize=7, color="#0b0b0b")
        ax.set_title(m); ax.set_xlim(0, 1.15); ax.set_yticks(y, ORDER); ax.invert_yaxis(); ax.grid(axis="y", visible=False)
    fig.suptitle("Test-set ranking quality on real Upwork gigs and resumes (mean ± 95% CI over samples)", y=1.03, fontsize=10)
    return fig


def fig_lsa(df):
    import matplotlib.pyplot as plt
    g = df.groupby("k")["nDCG@10"].agg(["mean", "std"]).reset_index()
    fig, ax = plt.subplots(figsize=(4.6, 3))
    ax.errorbar(g.k, g["mean"], yerr=g["std"], marker="D", color=plotting.METHOD_COLORS["LSA"], capsize=2, label="LSA (rank k)")
    ax.axhline(df["tfidf_nDCG@10"].mean(), color=plotting.METHOD_COLORS["TF-IDF"], ls="--", lw=1.5, label="TF-IDF (full rank)")
    ax.set_xscale("log")
    ax.set_xlabel("latent dimension k"); ax.set_ylabel("nDCG@10"); ax.set_title("Effect of LSA rank k"); ax.legend()
    return fig


def fig_svd(df):
    import matplotlib.pyplot as plt
    g = df.groupby("n_iter")[["max_rel_err", "mean_rel_err"]].agg(["mean", "max"])
    fig, ax = plt.subplots(figsize=(4.6, 3))
    ax.semilogy(g.index, g[("max_rel_err", "mean")], marker="o", color=plotting.PALETTE[0], label="max_i relative error")
    ax.semilogy(g.index, g[("mean_rel_err", "mean")], marker="s", color=plotting.PALETTE[1], label="mean_i relative error")
    ax.axhline(1e-3, color="#8a8984", ls=":", lw=1.2, label="tolerance 1e-3")
    ax.set_xlabel("power iterations q (n_iter)"); ax.set_ylabel(r"$|\hat\sigma_i-\sigma_i|/\sigma_i$")
    ax.set_title(f"Randomized SVD convergence (k = {LSA_K})"); ax.legend()
    return fig


def fig_robustness(df):
    import matplotlib.pyplot as plt
    fig, axes = plt.subplots(1, 2, figsize=(9, 3.1), sharey=True)
    labels = {"resume_words": "words kept from each resume (log scale)", "tag_drop_prob": "probability of deleting each seeker tag"}
    for ax, f in zip(axes, ["resume_words", "tag_drop_prob"]):
        sub = df[df.factor == f].copy()
        if f == "resume_words":
            full = int(np.ceil(_base()[1].bio.str.split().str.len().max()))
            sub["x"] = sub.level.replace("all", full).astype(float)
        else:
            sub["x"] = sub.level.astype(float)
        for m in ORDER:
            g = sub[sub.method == m].groupby("x")["nDCG@10"].mean()
            ax.plot(g.index, g.values, marker=plotting.MARKERS[m], color=plotting.METHOD_COLORS[m], label=m,
                    lw=2.6 if m == "Hybrid" else 1.6)
        if f == "resume_words":
            ax.set_xscale("log")
        ax.set_xlabel(labels[f]); ax.set_title("Robustness: " + ("shorter resumes" if f == "resume_words" else "missing tags"))
    axes[0].set_ylabel("nDCG@10")
    axes[1].legend(loc="center left", bbox_to_anchor=(1.0, 0.5))
    return fig


def fig_grid(grid):
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(4.8, 3))
    for text, col in (("TF-IDF", plotting.METHOD_COLORS["TF-IDF"]), ("LSA", plotting.METHOD_COLORS["LSA"])):
        g = grid[grid.text == text]
        ax.plot(g.w_text, g["val_nDCG@10"], marker="o", color=col, label=f"Hybrid with {text} text")
    ax.set_xlabel(r"text weight $\alpha$ (0 = tags only, 1 = text only)"); ax.set_ylabel("validation nDCG@10")
    ax.set_title("Choosing the hybrid weight (sample 0)"); ax.legend()
    return fig


def fig_scaling(df):
    import matplotlib.pyplot as plt
    g = df.groupby("n").median(numeric_only=True)
    fig, ax = plt.subplots(figsize=(4.8, 3.1))
    cols = {"TF-IDF scoring": plotting.PALETTE[2], "Jaccard scoring": plotting.PALETTE[1],
            "Hungarian assignment": plotting.PALETTE[0], "Greedy assignment": plotting.PALETTE[6]}
    slopes = {}
    for c, col in cols.items():
        ax.loglog(g.index, g[c], marker="o", color=col, label=c)
        slopes[c] = float(np.polyfit(np.log(g.index[2:]), np.log(g[c].iloc[2:]), 1)[0])
    ax.set_xlabel("n gigs = n seekers"); ax.set_ylabel("seconds (median of 3)"); ax.set_title("Computational cost")
    ax.legend()
    return fig, slopes


def fig_mc(df):
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(4.6, 3))
    for m in ["Hybrid", "Jaccard (tags)", "TF-IDF"]:
        x = df[df.method == m].sort_values("seed")["nDCG@10"].to_numpy()
        n = np.arange(1, len(x) + 1)
        mu = np.cumsum(x) / n
        sd = np.array([x[:i].std(ddof=1) if i > 1 else 0 for i in n])
        half = np.where(n > 1, stats.t.ppf(0.975, np.maximum(n - 1, 1)) * sd / np.sqrt(n), np.nan)
        ax.plot(n, mu, color=plotting.METHOD_COLORS[m], label=m)
        ax.fill_between(n, mu - half, mu + half, color=plotting.METHOD_COLORS[m], alpha=0.15, lw=0)
    ax.xaxis.set_major_locator(plt.MaxNLocator(integer=True))
    ax.set_xlabel("number of random samples R"); ax.set_ylabel("running mean nDCG@10")
    ax.set_title("Convergence of the Monte Carlo estimate"); ax.legend()
    return fig


def fig_categories(per_cat, top=15):
    import matplotlib.pyplot as plt
    g = per_cat.groupby("category").agg(n=("Hybrid", "size"), Hybrid=("Hybrid", "mean"), TFIDF=("TF-IDF", "mean"))
    g = g[g.n >= 20].sort_values("n", ascending=False).head(top).sort_values("Hybrid")
    fig, ax = plt.subplots(figsize=(5.6, 4))
    y = np.arange(len(g))
    ax.barh(y, g.Hybrid, color=plotting.METHOD_COLORS["Hybrid"], height=0.6, label="Hybrid")
    ax.scatter(g.TFIDF, y, color=plotting.METHOD_COLORS["TF-IDF"], zorder=3, s=14, label="TF-IDF")
    ax.set_yticks(y, [f"{c} ({n})" for c, n in zip(g.index, g.n)]); ax.set_xlim(0, 1)
    ax.set_xlabel("mean test nDCG@10"); ax.set_title("Ranking quality by Upwork category"); ax.legend(loc="lower right")
    ax.grid(axis="y", visible=False)
    return fig


def save(fig, name):
    import matplotlib.pyplot as plt
    FIGS.mkdir(exist_ok=True)
    fig.savefig(FIGS / f"{name}.pdf")
    fig.savefig(FIGS / f"{name}.png")
    plt.close(fig)


# ---------------------------------------------------------------------------
def main(argv=None):
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--quick", action="store_true", help="3 samples and smaller sweeps (smoke test)")
    args = ap.parse_args(argv)
    if not data_available():
        sys.exit("Raw data not found. Download the two Kaggle files into data/raw/ (see README.md).")
    plotting.setup()
    RESULTS.mkdir(exist_ok=True)
    n_main = 3 if args.quick else 20
    n_side = 2 if args.quick else 5
    seeds = list(range(n_main))
    side = list(range(n_side))
    t_start = time.perf_counter()

    print(f"[E1] main comparison over {n_main} random samples", flush=True)
    per_seed, chosen, assign, ablation, grid0, per_cat = run_main(seeds)
    per_seed.to_csv(RESULTS / "main_metrics_per_seed.csv", index=False)
    chosen.to_csv(RESULTS / "hybrid_selected_params.csv", index=False)
    grid0.to_csv(RESULTS / "weight_grid_seed0.csv", index=False)
    metrics = ["P@5", "R@100", "nDCG@10", "MRR"]
    summary = summarise(per_seed, "method", metrics)
    summary.to_csv(RESULTS / "main_metrics_summary.csv", index=False)
    ablation_sum = summarise(ablation, "variant", metrics)
    ablation_sum.to_csv(RESULTS / "ablation_summary.csv", index=False)
    assign.to_csv(RESULTS / "assignment_per_seed.csv", index=False)
    assign_sum = assign.groupby(["market", "policy"], sort=False).mean(numeric_only=True).drop(columns="seed").reset_index()
    assign_sum.to_csv(RESULTS / "assignment_summary.csv", index=False)
    cat_sum = per_cat.groupby("category").agg(n_test_gigs=("Hybrid", "size"), **{
        f"{m}_nDCG@10": (m, "mean") for m in ("Hybrid", "TF-IDF", "Jaccard (tags)")}).sort_values("n_test_gigs", ascending=False)
    cat_sum.to_csv(RESULTS / "per_category_ndcg.csv")
    tests = [paired_tests(per_seed, "Hybrid", b, m) for b in ("Jaccard (tags)", "TF-IDF", "LSA") for m in ("nDCG@10", "P@5")]
    pd.DataFrame(tests).to_csv(RESULTS / "significance_tests.csv", index=False)

    # Most common tuned configuration -> used for the robustness sweep.
    mode = chosen.groupby(["text", "w_text"]).size().idxmax()
    params = {"text": mode[0], "w_text": float(mode[1])}

    print("[E2] LSA rank sweep", flush=True); lsa = run_lsa_sweep(side); lsa.to_csv(RESULTS / "lsa_rank_sweep.csv", index=False)
    print("[E3] randomized SVD convergence", flush=True); svd_df, svd_meta = run_svd_convergence(); svd_df.to_csv(RESULTS / "svd_convergence.csv", index=False)
    print("[E4] robustness", flush=True); rob = run_robustness(side, params); rob.to_csv(RESULTS / "robustness.csv", index=False)
    print("[E7] runtime scaling", flush=True)
    scaling = run_scaling(sizes=(100, 200, 400, 800) if args.quick else (100, 200, 400, 800, 1600, 2400))
    scaling.to_csv(RESULTS / "runtime_scaling.csv", index=False)

    save(fig_main(summary), "fig_main_metrics")
    save(fig_lsa(lsa), "fig_lsa_rank")
    save(fig_svd(svd_df), "fig_svd_convergence")
    save(fig_robustness(rob), "fig_robustness")
    save(fig_grid(grid0), "fig_weight_grid")
    fs, slopes = fig_scaling(scaling); save(fs, "fig_runtime_scaling")
    save(fig_mc(per_seed), "fig_monte_carlo")
    save(fig_categories(per_cat), "fig_per_category")

    gigs_all, seekers_all, cmap, vocab = _base()
    s0, g0, rel0 = sample(RealConfig(seed=0))
    n_tags = seekers_all.declared_skills.map(lambda x: len(split_set(x)))
    meta = {
        "config": RealConfig().__dict__, "n_main_samples": n_main, "n_side_samples": n_side,
        "lsa_k": LSA_K, "svd_n_iter": SVD_ITERS, "w_text_grid": W_TEXT_GRID.tolist(), "n_scarce": N_SCARCE,
        "selected_params_mode": params,
        "selected_params_counts": chosen.groupby(["text", "w_text"]).size().rename("count").reset_index().to_dict("records"),
        "svd": svd_meta, "runtime_loglog_slopes_n>=400": slopes,
        "data": {
            "gigs_raw": int(len(load_gigs())), "gigs_after_mapping": int(len(gigs_all)), "upwork_categories_mapped": int(len(cmap)),
            "seekers": int(len(seekers_all)), "resume_categories": int(seekers_all.category.nunique()),
            "skill_vocabulary": int(len(vocab)),
            "gig_tags_median": float(gigs_all.required_skills.map(lambda x: len(split_set(x))).median()),
            "seeker_tags_median": float(n_tags.median()),
            "resume_words_median": float(seekers_all.bio.str.split().str.len().median()),
            "sample0_relevance_distribution": (np.bincount(rel0.ravel(), minlength=3) / rel0.size).tolist(),
            "sample0_mean_rel2_per_gig": float((rel0 >= 2).sum(1).mean()),
            "sample0_vocabulary_size": int(len(fit_tfidf(g0.description, s0.bio)[0].vocabulary_)),
        },
        "total_runtime_seconds": time.perf_counter() - t_start,
        "numpy": np.__version__, "pandas": pd.__version__,
    }
    import scipy, sklearn, matplotlib
    meta.update({"scipy": scipy.__version__, "sklearn": sklearn.__version__, "matplotlib": matplotlib.__version__})
    (RESULTS / "run_metadata.json").write_text(json.dumps(meta, indent=2, default=float))
    from .make_tables import main as make_tables
    make_tables()
    print(summary.round(3).to_string(index=False))
    print(f"done in {meta['total_runtime_seconds']:.1f}s")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile giglink_ai/demo.py
"""Command-line demo: type a gig and see which real resumes GigLink would suggest.

Example::

    python -m giglink_ai.demo --text "Need a logo and brand kit for a small coffee shop" \
        --tags "logo design;graphic design;adobe illustrator"

The seeker pool is the 2,483 real resumes in data/raw/Resume.csv.
"""

from __future__ import annotations

import argparse

import numpy as np
import pandas as pd

from .models import hybrid_scores, jaccard_scores, lsa_scores
from .real_data import _base

# Most frequently selected configuration in results/hybrid_selected_params.csv
DEFAULT_W_TEXT = 0.2


def recommend(gig: dict, seekers: pd.DataFrame, top: int = 5, w_text=DEFAULT_W_TEXT) -> pd.DataFrame:
    gigs = pd.DataFrame([gig])
    text = lsa_scores(gigs.description, seekers.bio, k=40)
    tags = jaccard_scores(gigs.required_skills.str.lower(), seekers.declared_skills)
    H = hybrid_scores(text, tags, np.zeros(len(seekers)), w_text, 0.0)[0]
    best = np.argsort(-H)[:top]
    out = seekers.iloc[best][["seeker_id", "category"]].copy()
    out.insert(1, "score", H[best].round(3))
    out.insert(2, "text_sim", text[0, best].round(3))
    out.insert(3, "tag_jaccard", tags[0, best].round(3))
    out["resume_start"] = seekers.bio.iloc[best].str.slice(0, 70).to_numpy()
    return out


def main(argv=None):
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--text", default="Need a logo and brand kit for a small coffee shop opening next month")
    ap.add_argument("--tags", default="logo design;graphic design;adobe illustrator;brand identity",
                    help="';'-separated Upwork-style skill tags")
    ap.add_argument("--top", type=int, default=5)
    a = ap.parse_args(argv)
    _, seekers, _, _ = _base()
    gig = {"gig_id": "DEMO", "description": a.text, "required_skills": a.tags}
    pd.set_option("display.width", 200, "display.max_colwidth", 70)
    print(f"Gig: {a.text}\nTags: {a.tags}\n")
    print(recommend(gig, seekers, a.top).to_string(index=False))


if __name__ == "__main__":
    main()

## Run the experiments

In [ ]:
!python -m giglink_ai.experiments

## Results

In [ ]:
import pandas as pd
from IPython.display import Image, display

summary = pd.read_csv("results/main_metrics_summary.csv").set_index("method")
display(summary[["P@5_mean", "R@100_mean", "nDCG@10_mean", "MRR_mean"]].round(3))
display(pd.read_csv("results/significance_tests.csv").round(4))
display(pd.read_csv("results/assignment_summary.csv").round(3))

for path in sorted(glob.glob("figures/*.png")):
    print(os.path.basename(path))
    display(Image(path, width=800))

## Try the matcher

Edit `description` and `skills` to test any type of work, then run the cell. Examples:
- `"Need a tutor for college algebra and calculus"` / `"math tutoring;algebra;calculus"`
- `"Edit short TikTok videos for our online shop"` / `"video editing;adobe premiere pro;tiktok"`
- `"Encode 500 receipts into an Excel sheet"` / `"data entry;microsoft excel"`

In [ ]:
from giglink_ai.demo import recommend
from giglink_ai.real_data import _base

description = "Need a logo and brand kit for a small coffee shop opening next month"
skills = "logo design;graphic design;adobe illustrator;brand identity"

_, seekers, _, _ = _base()
recommend({"gig_id": "MY-GIG", "description": description, "required_skills": skills}, seekers, top=5)